# 1. Set up parameters for ASL

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from obspy import read_inventory
from flovopy.asl.wrappers import run_single_event, find_event_files, run_all_events
from flovopy.processing.sam import VSAM, DSAM 
from flovopy.asl.config import ASLConfig, tweak_config
# -------------------------- Config --------------------------
import asl_env

# parameters for envelopes and cross-correlation
SMOOTH_SECONDS  = 1.0
MAX_LAG_SECONDS = 8.0
MIN_XCORR       = 0.5

# other parameters
DIST_MODE = "3d" # or 2d. will essentially squash Montserrat topography and stations onto a sea-level plane, ignored elevation data, e.g. for computing distances

# Inventory of Montserrat stations
INV = asl_env.INV
print(f"[INV] Networks: {len(INV)}  Stations: {sum(len(n) for n in INV)}  Channels: {sum(len(sta) for net in INV for sta in net)}")

# Montserrat station corrections estimated from regionals
station_corrections_csv = asl_env.STATION_CORRECTIONS_DIR / "station_gains_intervals.csv"
annual_station_corrections_csv = asl_env.STATION_CORRECTIONS_DIR / "station_gains_intervals_by_year.csv"
station_corrections_df = pd.read_csv(station_corrections_csv)
annual_station_corrections_df = pd.read_csv(annual_station_corrections_csv)

# Montserrat pre-defined Grid (from 02 tutorial)
from flovopy.asl.grid import Grid
gridobj = Grid.load(asl_env.GRIDFILE_DEFAULT)
print(gridobj)
landgridobj = Grid.load(asl_env.INPUT_DIR / "land" / "Grid_9c2fd59b.pkl")

# Montserrat constants
print("Dome (assumed source) =", asl_env.DOME_LOCATION)

# events and wrappers
event_files = list(find_event_files(asl_env.INPUT_DIR))
eventcsvfile = Path(asl_env.OUTPUT_DIR) / "mseed_files.csv"
if not eventcsvfile.is_file():
    rows = [{"num": num, "f": str(f)} for num, f in enumerate(event_files)]
    df = pd.DataFrame(rows)
    df.to_csv(eventcsvfile, index=False)
best_file_nums  = [35, 36, 40, 52, 82, 83, 84, 116, 310, 338]
best_event_files = [event_files[i] for i in best_file_nums]
print(f'Best miniseed files are: {best_event_files}')
REFINE_SECTOR = False   # enable triangular dome-to-sea refinement

# Parameters to pass for making pygmt topo maps
topo_kw = {
    "inv": INV,
    "add_labels": True,
    "cmap": "gray",
    "region": asl_env.REGION_DEFAULT,
    "dem_tif": asl_env.DEM_DEFAULT,  # basemap shading from your GeoTIFF - but does not actually seem to use this unless topo_color=True and cmap=None
    "frame": True,
    "dome_location": asl_env.DOME_LOCATION,
}

[INV] Networks: 1  Stations: 48  Channels: 77
[INFO] Grid loaded from /Users/glennthompson/Developer/CompSciS26/asl/metadata/MASTER_GRID_MONTSERRAT.pkl
[GRID] 831x1051 nodes (873381 total)  spacing=10.0 m  [mask: 59905/873381 nodes kept (6.9%)]
[INFO] Grid loaded from /Volumes/classdata/ASL_inputs/land/Grid_9c2fd59b.pkl
Dome (assumed source) = {'lat': 16.7106, 'lon': -62.17747, 'elev': 1000.0}
Best miniseed files are: ['/Volumes/classdata/ASL_inputs/biggest_pdc_events/2000-08-05-1840-17S.MVO___019.cleaned', '/Volumes/classdata/ASL_inputs/biggest_pdc_events/2000-08-07-0441-43S.MVO___019.cleaned', '/Volumes/classdata/ASL_inputs/biggest_pdc_events/2000-09-14-1900-58S.MVO___019.cleaned', '/Volumes/classdata/ASL_inputs/biggest_pdc_events/2000-11-26-2123-08S.MVO___019.cleaned', '/Volumes/classdata/ASL_inputs/biggest_pdc_events/2006-04-12-0025-40S.MVO___031.cleaned', '/Volumes/classdata/ASL_inputs/biggest_pdc_events/2006-04-13-1534-00S.MVO___031.cleaned', '/Volumes/classdata/ASL_inputs/bigges

# Build a baseline configuration
This is inherited by various downstream functions
This describes the physical parameters, the station metadata, the grid, the misfit algorithm, etc.

In [2]:
DEBUG=False
baseline_cfg = ASLConfig(
    inventory=asl_env.INV,
    output_base=asl_env.OUTPUT_DIR,
    gridobj=gridobj,
    global_cache=asl_env.GLOBAL_CACHE,
    station_correction_dataframe=station_corrections_df,
    wave_kind="surface",
    speed=1.5,
    Q=23, 
    peakf=2.0,
    dist_mode="3d", 
    misfit_engine="r2",
    window_seconds=5.0,
    min_stations=5,
    sam_class=VSAM, 
    sam_metric="mean",
    debug=DEBUG,
)
baseline_cfg.build()

[ASLConfig.build] Computing/loading station→node distances …
[COMPUTE OR LOAD DISTANCES] Computing fresh distances…
[ASLConfig.build] Computing/loading amplitude corrections …


ASLConfig(inventory=<obspy.core.inventory.inventory.Inventory object at 0x178ceda90>, output_base=PosixPath('/Users/glennthompson/compsci_asl/asl_global_cache'), gridobj=<flovopy.asl.grid.Grid object at 0x177faa990>, global_cache=PosixPath('/Users/glennthompson/compsci_asl/asl_global_cache'), wave_kind='surface', station_correction_dataframe=    start_time  end_time       seed_id      gain  \
0          NaN       NaN  MV.MBBY..BHE  0.951905   
1          NaN       NaN  MV.MBBY..BHN  0.956411   
2          NaN       NaN  MV.MBBY..BHZ  1.000000   
3          NaN       NaN  MV.MBBY..HHE  1.000000   
4          NaN       NaN  MV.MBBY..HHN  1.000000   
5          NaN       NaN  MV.MBBY..HHZ  0.884720   
6          NaN       NaN  MV.MBFL..EHZ  0.506110   
7          NaN       NaN  MV.MBFR..HHE  1.905681   
8          NaN       NaN  MV.MBFR..HHN  1.936456   
9          NaN       NaN  MV.MBFR..HHZ  0.935610   
10         NaN       NaN  MV.MBGB..BHE  1.000000   
11         NaN       NaN  MV.MBG

# Run one event with this baseline configuration

In [3]:
DEBUG=True
result = run_single_event(
    mseed_file=event_files[0],
    cfg=baseline_cfg,
    refine_sector=False,
    station_gains_df=None,
    switch_event_ctag = True,
    topo_kw=topo_kw,
    mseed_units='m/s', # default units for miniseed files being used - probably "Counts" or "m/s"        
    reduce_time=True,
    debug=DEBUG,
)


[ASL] Running event: /Volumes/classdata/ASL_inputs/biggest_pdc_events/2000-01-12-1208-16S.MVO___019.cleaned with config tag=VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC
[ASL] Config outdir: /Users/glennthompson/compsci_asl/asl_global_cache/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC
[GAINS] No interval filtering (time-agnostic gains table). Applying global gains.
[GAINS] Applied gains to 6 traces.
[GAINS] Interval used: NaT → NaT | corrected 6 traces; missing 0
6 Trace(s) in Stream:
MV.MBRY..BHZ | 2000-01-12T12:08:14.447551Z - 2000-01-12T12:10:12.034217Z | 75.0 Hz, 8820 samples
MV.MBSS..SHZ | 2000-01-12T12:08:14.441512Z - 2000-01-12T12:10:12.028178Z | 75.0 Hz, 8820 samples
MV.MBWH..SHZ | 2000-01-12T12:08:14.449868Z - 2000-01-12T12:10:12.036535Z | 75.0 Hz, 8820 samples
MV.MBGB..BHZ | 2000-01-12T12:08:14.445479Z - 2000-01-12T12:10:12.032146Z | 75.0 Hz, 8820 samples
MV.MBMH..SHZ | 2000-01-12T12:08:14.450499Z - 2000-01-12T12:10:12.037166Z | 75.0 Hz, 8820 samples
MV.MBLG..SHZ | 2000-01-12T12:08

/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will 

[ASL] fast_locate: done.
[ASL] Location complete.
source sanity check:
               lat         lon          DR      misfit       azgap   nsta  \
count  117.000000  117.000000  117.000000  117.000000  117.000000  117.0   
mean    16.707498  -62.176458   52.562922    0.142703  154.608781    6.0   
std      0.007338    0.005826   51.822068    0.071624   26.824964    0.0   
min     16.681282  -62.195409   15.576184    0.017335  120.410565    6.0   
25%     16.704125  -62.179446   19.820734    0.083693  137.464272    6.0   
50%     16.710150  -62.176536   23.698080    0.129302  147.412069    6.0   
75%     16.712399  -62.172498   66.535385    0.193590  167.540768    6.0   
max     16.715816  -62.165925  194.805034    0.343677  247.884205    6.0   

          node_index  connectedness  
count     117.000000   1.170000e+02  
mean   400626.820513   6.076154e-01  
std     85798.028705   1.114998e-16  
min     94064.000000   6.076154e-01  
25%    361237.000000   6.076154e-01  
50%    431667.0

/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/obspy/core/inventory/network.py:321: UserWarning: Found more than one matching channel metadata. Returning first.
  warnings.warn(msg)
text [WARNING]: Skipped 1 records as blank - please check input data.


[MAP] Adding title ASL event 2000/01/12 12:08:14, 117 s

VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC
ML=1.38, ME=1.50
[2026-04-15 21:57:59] [ASL:PLOT] saved figure: /Users/glennthompson/compsci_asl/asl_global_cache/2000-01-12-1208-16S.MVO___019/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC/map_VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC.png
[ASL:SOURCE_TO_CSV] Writing source to CSV…
[ASL] Source written to CSV: /Users/glennthompson/compsci_asl/asl_global_cache/2000-01-12-1208-16S.MVO___019/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC/source_VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC.csv
[ASL:PLOT_REDUCED_DISPLACEMENT/VELOCITY]
[ASL:PLOT_MISFIT]
[ASL:PLOT_MISFIT_HEATMAP]
[2026-04-15 21:57:59] [MISFIT] peak DR index = 20
Dataframe with 117 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 117 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 117 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 117 rows is already on a reg

/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/obspy/core/inventory/network.py:321: UserWarning: Found more than one matching channel metadata. Returning first.
  warnings.warn(msg)
makecpt [WARNING]: Without inc in -T option, -Z has no effect (ignored)


[MAP] Adding title Misfit heatmap (peak DR time)
[2026-04-15 21:58:00] [MISFIT] saved: /Users/glennthompson/compsci_asl/asl_global_cache/2000-01-12-1208-16S.MVO___019/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC/misfit_heatmap_VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC.png
[CHECK:SRC] lat: shape=(117,) dtype=float64 finite%=100.0
[CHECK:SRC] lon: shape=(117,) dtype=float64 finite%=100.0
[CHECK:SRC] DR: shape=(117,) dtype=float64 finite%=100.0
[CHECK:SRC] misfit: shape=(117,) dtype=float64 finite%=100.0
[CHECK:SRC] t: shape=(117,) dtype=object finite%=nan
[CHECK:SRC] nsta: shape=(117,) dtype=int64 finite%=100.0
[ASL] Single-event summary: {'tag': 'VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC', 'outdir': '/Users/glennthompson/compsci_asl/asl_global_cache/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC', 'event_dir': '/Users/glennthompson/compsci_asl/asl_global_cache/2000-01-12-1208-16S.MVO___019', 'log_file': '/Users/glennthompson/compsci_asl/asl_global_cache/2000-01-12-1208-16S.MVO___019/VSAM_mean_5

# Run events (one event=one miniseed file) with this baseline configuration

In [4]:
summaries = []
REFINE_SECTOR=False
for i, ev in zip(best_file_nums, best_event_files):
    print(f"[{i}/{len(event_files)}] {ev}")
    result = run_single_event(
        mseed_file=str(ev),
        cfg=baseline_cfg,
        refine_sector=REFINE_SECTOR,
        station_gains_df=None,
        switch_event_ctag = True,
        topo_kw=topo_kw,
        mseed_units='m/s', # default units for miniseed files being used - probably "Counts" or "m/s"        
        reduce_time=True,
        debug=DEBUG,
    )
    summaries.append(result)

# Summarize
df = pd.DataFrame(summaries)
display(df)

summary_csv = Path(asl_env.OUTPUT_DIR) / f"{baseline_cfg.tag()}__summary.csv"
df.to_csv(summary_csv, index=False)
print(f"Summary saved to: {summary_csv}")

if not df.empty:
    n_ok = int((~df.get("error").notna()).sum()) if "error" in df.columns else len(df)
    print(f"Success: {n_ok}/{len(df)}")

[35/368] /Volumes/classdata/ASL_inputs/biggest_pdc_events/2000-08-05-1840-17S.MVO___019.cleaned

[ASL] Running event: /Volumes/classdata/ASL_inputs/biggest_pdc_events/2000-08-05-1840-17S.MVO___019.cleaned with config tag=VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC
[ASL] Config outdir: /Users/glennthompson/compsci_asl/asl_global_cache/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC
[GAINS] No interval filtering (time-agnostic gains table). Applying global gains.
[GAINS] Applied gains to 7 traces.
[GAINS] Interval used: NaT → NaT | corrected 7 traces; missing 0
7 Trace(s) in Stream:
MV.MBRY..BHZ | 2000-08-05T18:40:15.194217Z - 2000-08-05T18:45:13.034217Z | 75.0 Hz, 22339 samples
MV.MBSS..SHZ | 2000-08-05T18:40:15.188178Z - 2000-08-05T18:45:13.028178Z | 75.0 Hz, 22339 samples
MV.MBBY..BHZ | 2000-08-05T18:40:15.193052Z - 2000-08-05T18:45:13.033052Z | 75.0 Hz, 22339 samples
MV.MBGH..BHZ | 2000-08-05T18:40:15.190530Z - 2000-08-05T18:45:13.030530Z | 75.0 Hz, 22339 samples
MV.MBWH..SHZ | 2000-08-05T18

/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4786: UserWarning: Warning: 'partition' will 

Dataframe with 297 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 297 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 297 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 297 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 297 rows is already on a regular 1 s grid based on column 'time'

metric:  median
MV.MBRY..BHZ, median: 4.016e+02, station correction: 0.424
MV.MBSS..SHZ, median: 1.452e+02, station correction: 1.172
MV.MBBY..BHZ, median: 1.222e+02, station correction: 1.392
MV.MBGH..BHZ, median: 1.597e+02, station correction: 1.065
MV.MBWH..SHZ, median: 2.094e+02, station correction: 0.812
MV.MBGB..BHZ, median: 1.541e+02, station correction: 1.104
MV.MBMH..SHZ, median: 8.576e+01, station correction: 1.983
network: median: 1.701e+02, station correction std: 4.642e-01
Dataframe with 297 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 297 rows is a

/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/obspy/core/inventory/network.py:321: UserWarning: Found more than one matching channel metadata. Returning first.
  warnings.warn(msg)
text [WARNING]: Skipped 1 records as blank - please check input data.


[MAP] Adding title ASL event 2000/08/05 18:40:15, 297 s

VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC
ML=1.97, ME=2.58
[2026-04-15 21:58:04] [ASL:PLOT] saved figure: /Users/glennthompson/compsci_asl/asl_global_cache/2000-08-05-1840-17S.MVO___019/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC/map_VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC.png
[ASL:SOURCE_TO_CSV] Writing source to CSV…
[ASL] Source written to CSV: /Users/glennthompson/compsci_asl/asl_global_cache/2000-08-05-1840-17S.MVO___019/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC/source_VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC.csv
[ASL:PLOT_REDUCED_DISPLACEMENT/VELOCITY]
[ASL:PLOT_MISFIT]
[ASL:PLOT_MISFIT_HEATMAP]
[2026-04-15 21:58:04] [MISFIT] peak DR index = 43
Dataframe with 297 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 297 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 297 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 297 rows is already on a reg

...

[MAP] Adding title ASL event 1999/07/11 19:16:36, 117 s

VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC
ML=1.79, ME=1.92
[2026-04-15 21:58:43] [ASL:PLOT] saved figure: /Users/glennthompson/compsci_asl/asl_global_cache/9907-11-1916-38S.MVO_19_1/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC/map_VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC.png
[ASL:SOURCE_TO_CSV] Writing source to CSV…
[ASL] Source written to CSV: /Users/glennthompson/compsci_asl/asl_global_cache/9907-11-1916-38S.MVO_19_1/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC/source_VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC.csv
[ASL:PLOT_REDUCED_DISPLACEMENT/VELOCITY]
[ASL:PLOT_MISFIT]
[ASL:PLOT_MISFIT_HEATMAP]
[2026-04-15 21:58:43] [MISFIT] peak DR index = 39
Dataframe with 117 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 117 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 117 rows is already on a regular 1 s grid based on column 'time'
Dataframe with 117 rows is already on a regular 1 s

/Users/glennthompson/miniconda3/envs/flovopy_env/lib/python3.12/site-packages/obspy/core/inventory/network.py:321: UserWarning: Found more than one matching channel metadata. Returning first.
  warnings.warn(msg)
makecpt [WARNING]: Without inc in -T option, -Z has no effect (ignored)


[MAP] Adding title Misfit heatmap (peak DR time)
[2026-04-15 21:58:44] [MISFIT] saved: /Users/glennthompson/compsci_asl/asl_global_cache/9907-11-1916-38S.MVO_19_1/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC/misfit_heatmap_VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC.png
[CHECK:SRC] lat: shape=(117,) dtype=float64 finite%=100.0
[CHECK:SRC] lon: shape=(117,) dtype=float64 finite%=100.0
[CHECK:SRC] DR: shape=(117,) dtype=float64 finite%=100.0
[CHECK:SRC] misfit: shape=(117,) dtype=float64 finite%=100.0
[CHECK:SRC] t: shape=(117,) dtype=object finite%=nan
[CHECK:SRC] nsta: shape=(117,) dtype=int64 finite%=100.0
[ASL] Single-event summary: {'tag': 'VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC', 'outdir': '/Users/glennthompson/compsci_asl/asl_global_cache/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC', 'event_dir': '/Users/glennthompson/compsci_asl/asl_global_cache/9907-11-1916-38S.MVO_19_1', 'log_file': '/Users/glennthompson/compsci_asl/asl_global_cache/9907-11-1916-38S.MVO_19_1/VSAM_mean_5s_surface_v1

,tag,outdir,event_dir,log_file,outputs,elapsed_s
0,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,5.52
1,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,5.80
2,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,5.64
3,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,4.60
4,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,4.09
5,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,4.17
6,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,3.88
7,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,3.51
8,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,3.40
9,VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,/Users/glennthompson/compsci_asl/asl_global_ca...,{'primary': {'qml': '/Users/glennthompson/comp...,3.68


Summary saved to: /Users/glennthompson/compsci_asl/asl_global_cache/VSAM_mean_5s_surface_v1.5_Q23_F2_3d_r2_SC__summary.csv
Success: 10/10


run_single_event(
    mseed_file='/path/to/miniseed/file', 
    cfg=baseline_cfg, 
    reduce_time=True,
)

# Easily create new configurations that change 1 or 2 parameters
Here we create 18 new configurations, each is an entry in the changes dictionary.

In [5]:
variants = tweak_config(
    baseline_cfg,
    changes=[
        {"Q": 10},                                      # decrease Q from 23 to 10
        {"Q": 100},                                     # increase Q from 23 to 100
        {"speed": 0.5},                                 # decrease wave speed from 1.5 km/s to 0.5 km/s
        {"speed": 2.5},                                 # increase wave speed from 1.5 km/s to 2.5 km/s
        {"peakf": 8.0},                                 # increae peakf from 2.0 Hz to 8.0 Hz
        {"dist_mode": "2d"},                            # change from 3D to 2D: ignore terrain & ignore station elevations
        {"station_correction_dataframe": None},         # turn off station corrections
        {"gridobj":landgridobj},                        # try a grid that allows whole Southern end of island, not just dome & ravines
        {"misfit_engine": "l2"},                        # change the misfit function from r2 to l2
        {"misfit_engine": "lin"},                       # change the misfit function from r2 to lin
        {"window_seconds": 1.0},                        # decrease the moving window length from 5-s to 1-s
        {"sam_class": DSAM},                            # switch from VELOCITY Seismic Amplitude Measurement (VSAM) to DISPLACEMENT Seismic Amplitude Measurement (DSAM)
        {"sam_metric": "median"},                       # switch from MEAN of each 5-s moving time window, to MEDIAN
        {"sam_metric": "rms"},                          # switch from MEAN of each 5-s moving time window, to RMS
        {"sam_metric": "max"},                          # switch from MEAN of each 5-s moving time window, to MAX
        {"sam_metric": "LP"},                           # switch from MEAN in 0.5-18.0 Hz band to mean in LP band (0.5-4.0 Hz)
        {"sam_metric": "VT"},                           # switch from MEAN in 0.5-18.0 Hz band to mean in VT band (4.0-18.0 Hz)
        {"wave_kind": "body", "speed": 2.5},            # change multiple params: surface->body waves, wave speed 1.5->3.0 km/s - THIS IS A REFERENCE TO COMPARE SURFACE WAVES AND BODY WAVES
    ],
)
print(variants)

[ASLConfig.build] Computing/loading station→node distances …
[ASLConfig.build] Computing/loading amplitude corrections …
[ASLConfig.build] Computing/loading station→node distances …
[ASLConfig.build] Computing/loading amplitude corrections …
[ASLConfig.build] Computing/loading station→node distances …
[ASLConfig.build] Computing/loading amplitude corrections …
[ASLConfig.build] Computing/loading station→node distances …
[ASLConfig.build] Computing/loading amplitude corrections …
[ASLConfig.build] Computing/loading station→node distances …
[ASLConfig.build] Computing/loading amplitude corrections …
[ASLConfig.build] Computing/loading station→node distances …
[COMPUTE OR LOAD DISTANCES] Computing fresh distances…
[ASLConfig.build] Computing/loading amplitude corrections …
[ASLConfig.build] Computing/loading station→node distances …
[ASLConfig.build] Computing/loading amplitude corrections …
[ASLConfig.build] Computing/loading station→node distances …
[COMPUTE OR LOAD DISTANCES] Computing

# Run pairs: variant vs baseline

In [6]:
from flovopy.asl.compare import compare_runs

scored, summary, win_counts = compare_runs(
    baseline_cfg,
    events=best_event_files,
    variants=variants,
    run_single_event=run_single_event,
    refine_sector=False,
    topo_kw=topo_kw,
    run_if_missing_baseline=True,
    run_if_missing_variants=True,
    # the following numbers should add up to 1.0. if truly just want to see difference RELATIVE to baseline, set w_sep=1.0, and others to 0.0. for ABSOLUTE quality check, set w_sep to 0.0
    w_sep=0.5, # set this high to penalize large location difference from the baseline config
    w_misfit=0.2, # set this high to punish high misfits
    w_azgap=0.1, # punish larger azimuthal gaps
    w_conn=0.1, # reward more connectedness
    w_rough=0.1, # reward less roughness / more straightness
)

# Inspect/save
if scored is not None:
    display(summary)
    display(win_counts)
    summary.to_csv(asl_env.OUTPUT_DIR / "pairwise_summary_surface.csv", index=False)

ImportError: cannot import name 'compare_runs' from 'flovopy.asl.compare' (/Users/glennthompson/Developer/flovopy/flovopy/asl/compare.py)

In [ ]:
scored, summary, win_counts = compare_runs(
    baseline_cfg,
    events=best_event_files,
    variants=variants,
    run_single_event=run_single_event,
)


## baseline-free absolute scoring


In [ ]:
from flovopy.asl.compare import add_baseline_free_scores, summarize_absolute_runs, per_event_winner_abs, crawl_intrinsic_runs

# Option B: OR just crawl everything that already exists under OUTPUT_DIR
abs_tbl = crawl_intrinsic_runs(asl_env.OUTPUT_DIR)
weights = {
    "mean_misfit":     1.0,   # lower better
    "mean_azgap":      0.1,   # lower better
    "roughness_ratio": 0.1,   # lower better
    "connectedness":  -0.3,   # higher better (negative weight)
    "valid_frac":     -0.2,   # higher better (negative weight)
}
abs_scored = add_baseline_free_scores(abs_tbl, weights=weights)

abs_summary = summarize_absolute_runs(abs_scored)
winners_abs, win_counts_abs = per_event_winner_abs(abs_scored)

display(abs_summary)
display(win_counts_abs)

abs_summary.to_csv(asl_env.OUTPUT_DIR / "absolute_summary.csv", index=False)

# Now let's try to run a suite of body wave configurations


In [ ]:
# Create new baseline configuration - this time for body waves
baseline_cfg = ASLConfig(
    inventory=INV,
    output_base=asl_env.OUTPUT_DIR,
    gridobj=gridobj,
    global_cache=asl_env.GLOBAL_CACHE,
    station_correction_dataframe=station_corrections_df,
    wave_kind="body",
    speed=2.5,
    Q=23, 
    peakf=2.0,
    dist_mode="3d", 
    misfit_engine="r2",
    window_seconds=5.0,
    min_stations=5,
    sam_class=VSAM, 
    sam_metric="mean",
    debug=DEBUG,
)
baseline_cfg.build()

# Easily create new configurations - 18 new ones
variants = tweak_config(
    baseline_cfg,
    changes=[
        {"Q": 10},                                      # decrease Q from 23 to 10
        {"Q": 100},                                     # increase Q from 23 to 100
        {"speed": 1.5},                                 # decrease wave speed from 2.5 km/s to 1.5 km/s
        {"speed": 4.0},                                 # increase wave speed from 2.5 km/s to 4.0 km/s
        {"peakf": 8.0},                                 # increae peakf from 2.0 Hz to 8.0 Hz
        {"dist_mode": "2d"},                            # change from 3D to 2D: ignore terrain & ignore station elevations
        {"station_correction_dataframe": None},         # turn off station corrections
        {"gridobj":landgridobj},                        # try a grid that allows whole Southern end of island, not just dome & ravines
        {"misfit_engine": "l2"},                        # change the misfit function from r2 to l2
        {"misfit_engine": "lin"},                       # change the misfit function from r2 to lin
        {"window_seconds": 1.0},                        # decrease the moving window length from 5-s to 1-s
        {"sam_class": DSAM},                            # switch from VELOCITY Seismic Amplitude Measurement (VSAM) to DISPLACEMENT Seismic Amplitude Measurement (DSAM)
        {"sam_metric": "median"},                       # switch from MEAN of each 5-s moving time window, to MEDIAN
        {"sam_metric": "rms"},                          # switch from MEAN of each 5-s moving time window, to RMS
        {"sam_metric": "max"},                          # switch from MEAN of each 5-s moving time window, to MAX
        {"sam_metric": "LP"},                           # switch from MEAN in 0.5-18.0 Hz band to mean in LP band (0.5-4.0 Hz)
        {"sam_metric": "VT"},                           # switch from MEAN in 0.5-18.0 Hz band to mean in VT band (4.0-18.0 Hz)
        {"wave_kind": "surface", "speed": 1.0},         # change multiple params: surface->body waves, wave speed 1.5->3.0 km/s - THIS IS A REFERENCE TO COMPARE SURFACE WAVES AND BODY WAVES
    ],
)

# Run pairs - variant versus baseline
scored, summary, win_counts = compare_runs(
    baseline_cfg,
    events=best_event_files,
    variants=variants,
    run_single_event=run_single_event,
    refine_sector=False,
    topo_kw=topo_kw,
    run_if_missing_baseline=True,
    run_if_missing_variants=True,
    # the following numbers should add up to 1.0. if truly just want to see difference RELATIVE to baseline, set w_sep=1.0, and others to 0.0. for ABSOLUTE quality check, set w_sep to 0.0
    w_sep=0.5, # set this high to penalize large location difference from the baseline config
    w_misfit=0.2, # set this high to punish high misfits
    w_azgap=0.1, # punish larger azimuthal gaps
    w_conn=0.1, # reward more connectedness
    w_rough=0.1, # reward less roughness / more straightness
)

if scored is not None:
    display(summary)
    display(win_counts)
    summary.to_csv(asl_env.OUTPUT_DIR / "pairwise_summary_body.csv", index=False)


# baseline-free absolute scoring
abs_tbl = crawl_intrinsic_runs(asl_env.OUTPUT_DIR)
weights = {
    "mean_misfit":     1.0,   # lower better
    "mean_azgap":      0.1,   # lower better
    "roughness_ratio": 0.1,   # lower better
    "connectedness":  -0.3,   # higher better (negative weight)
    "valid_frac":     -0.2,   # higher better (negative weight)
}
abs_scored = add_baseline_free_scores(abs_tbl, weights=weights)

abs_summary = summarize_absolute_runs(abs_scored)
winners_abs, win_counts_abs = per_event_winner_abs(abs_scored)

display(abs_summary)
display(win_counts_abs)

abs_summary.to_csv(asl_env.OUTPUT_DIR / "absolute_summary.csv", index=False)

# Run all events demo

In [ ]:
# Replace the loop with one call
REFINE_SECTOR = True

summary_dir = run_all_events(
    input_dir=asl_env.INPUT_DIR,   # or a directory of files
    cfg=baseline_cfg,
    topo_kw=topo_kw,
    station_gains_df=None,
    refine_sector=REFINE_SECTOR,
    mseed_units="m/s",            # default units for your MSEED files
    reduce_time=True,
    switch_event_ctag=True,
    use_multiprocessing=False,    # set True if you want parallelism
    debug=DEBUG,
)

# Collect JSONL outputs into a DataFrame
summary_path = Path(summary_dir) / "summary.jsonl"
if summary_path.exists():
    df = pd.read_json(summary_path, lines=True)
    display(df)

    # Optional: also write a CSV copy
    summary_csv = asl_env.OUTPUT_DIR / f"{baseline_cfg.tag()}__summary.csv"
    df.to_csv(summary_csv, index=False)
    print(f"Summary saved to: {summary_csv}")

    if not df.empty:
        n_ok = int((~df.get("error").notna()).sum()) if "error" in df.columns else len(df)
        print(f"Success: {n_ok}/{len(df)}")

run_all_events(
    input_dir=INPUT_DIR,   
    cfg=baseline_cfg,
    use_multiprocessing=True,    
)

# Run same again, but with Land grid

In [ ]:
# Replace the loop with one call
DEBUG=False
LAND_DIR = Path(str(asl_env.OUTPUT_DIR).replace('ASL_RESULTS', 'ASL_RESULTS_LAND'))
baseline_cfg = ASLConfig(
    inventory=asl_env.INV,
    output_base=LAND_DIR,
    gridobj=landgridobj,
    global_cache=asl_env.GLOBAL_CACHE,
    station_correction_dataframe=station_corrections_df,
    wave_kind="surface",
    speed=1.5,
    Q=23,
    peakf=2.0,
    dist_mode="3d", 
    misfit_engine="r2",
    window_seconds=5.0,
    min_stations=5,
    sam_class=VSAM, 
    sam_metric="mean",
    debug=DEBUG,
)
baseline_cfg.build()
REFINE_SECTOR = True

summary_dir = run_all_events(
    input_dir=asl_env.INPUT_DIR,   # or a directory of files
    cfg=baseline_cfg,
    topo_kw=topo_kw,
    station_gains_df=None,
    refine_sector=REFINE_SECTOR,
    mseed_units="m/s",            # default units for your MSEED files
    reduce_time=True,
    switch_event_ctag=True,
    use_multiprocessing=False,    # set True if you want parallelism
    debug=DEBUG,
)

# Collect JSONL outputs into a DataFrame
summary_path = Path(summary_dir) / "summary.jsonl"
if summary_path.exists():
    df = pd.read_json(summary_path, lines=True)
    display(df)

    # Optional: also write a CSV copy
    summary_csv = Path(LAND_DIR) / f"{baseline_cfg.tag()}__summary.csv"
    df.to_csv(summary_csv, index=False)
    print(f"Summary saved to: {summary_csv}")

    if not df.empty:
        n_ok = int((~df.get("error").notna()).sum()) if "error" in df.columns else len(df)
        print(f"Success: {n_ok}/{len(df)}")